# Stage 1 Pretrain (Colab)
One-class VAE training on live data with Drive-mounted dataset.

In [ ]:

import os
import sys
import shutil
from pathlib import Path


use_colab = "google.colab" in sys.modules
if use_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    project_dir = Path('/content/drive/MyDrive/liveness_detection_vae')
else:
    project_dir = Path.cwd()


In [ ]:

import os
import sys
import shutil
from pathlib import Path

# project_dir is set in the previous cell
lib_dir = project_dir / 'lib'
sys.path.insert(0, str(lib_dir))
sys.path.insert(0, str(project_dir))

drive_root = project_dir / 'datasets'
drive_zip = drive_root / f"{DATASET_NAME}.zip"
local_root = Path('/content/datasets') if "google.colab" in sys.modules else drive_root
if "google.colab" in sys.modules:
    local_root.mkdir(parents=True, exist_ok=True)
local_zip = local_root / f"{DATASET_NAME}.zip"
extract_dir = local_root / DATASET_NAME
nested_dir = extract_dir / DATASET_NAME

def _has_npz(p: Path) -> bool:
    return p.is_dir() and any(p.rglob('*.npz'))

# Sync zip to local if available
if drive_zip.exists() and (not local_zip.exists() or drive_zip.stat().st_mtime > local_zip.stat().st_mtime):
    shutil.copy2(drive_zip, local_zip)

# Resolve data_dir (prefer local extracted; else extract local zip; else drive extracted)
if _has_npz(extract_dir):
    data_dir = extract_dir
elif _has_npz(nested_dir):
    data_dir = nested_dir
elif local_zip.exists():
    shutil.unpack_archive(str(local_zip), str(local_root))
    if _has_npz(nested_dir):
        data_dir = nested_dir
    elif _has_npz(extract_dir):
        data_dir = extract_dir
    else:
        data_dir = None
else:
    data_dir = None

if data_dir is None:
    drive_extract = drive_root / DATASET_NAME
    drive_nested = drive_extract / DATASET_NAME
    if _has_npz(drive_nested):
        data_dir = drive_nested
    elif _has_npz(drive_extract):
        data_dir = drive_extract

if data_dir is None:
    raise FileNotFoundError(f"Expected {DATASET_NAME} zip/extracted under {drive_root} or local {local_root}")

os.environ['BANDVAE_DATA_DIR'] = str(data_dir)

runs_dir = project_dir / 'runs'
runs_dir.mkdir(parents=True, exist_ok=True)
save_dir = runs_dir / 'stage1_pretrain_colab'
save_dir.mkdir(parents=True, exist_ok=True)
print(f'Save dir: {save_dir}')
print(f'Project dir: {project_dir}')
print(f'Data dir: {data_dir}')


In [ ]:

import time
import torch

import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from config_bandvae import get_config
from dataset_bandvae import LipLivenessBandDataset
from model_bandvae import BandSplitVAE, band_split_vae_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = get_config('full')
config.T_fixed = 300
config.fc_low = 2.0
config.fc_high = 8.0
config.filter_order = 4
config.C_h = 48
config.C_z = 12
config.dilations = [1, 2, 4]
config.lr = 1e-3
config.epochs = 20
config.batch_size = 64
config.num_workers = 4
config.data_dir = str(data_dir)
config.save_dir = str(save_dir)
config.device = device
print(config)


In [ ]:

dataset = LipLivenessBandDataset(
    data_dir=config.data_dir,
    T_fixed=config.T_fixed,
    fps=config.fps,
    use_procrustes=True,
    use_acceleration=config.use_acceleration,
    use_angle=config.use_angle,
    use_angle_rate=config.use_angle_rate,
    fc_low=config.fc_low,
    fc_high=config.fc_high,
    filter_order=config.filter_order,
    random_crop=True,
)

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_set,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print(f'Train: {len(train_set)}, Val: {len(val_set)}')


In [ ]:

model = BandSplitVAE(
    C_in_per_band=config.C_in_per_band,
    C_h=config.C_h,
    C_z=config.C_z,
    dilations=config.dilations,
).to(config.device)

optimizer = optim.Adam(model.parameters(), lr=config.lr)
use_amp = config.device == 'cuda'


In [ ]:


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_loss = lf_rec = bp_rec = hf_rec = 0.0
    n_batches = len(loader)

    for x_lf, x_bp, x_hf in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)

        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}

        loss, loss_dict = band_split_vae_loss(
            recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
        )

        total_loss += loss_dict['total']
        lf_rec += loss_dict['recon_lf']
        bp_rec += loss_dict['recon_bp']
        hf_rec += loss_dict['recon_hf']

    return {
        'total': total_loss / n_batches,
        'lf_rec': lf_rec / n_batches,
        'bp_rec': bp_rec / n_batches,
        'hf_rec': hf_rec / n_batches,
    }


def train_epoch(model, loader, optimizer, device, use_amp=False):
    model.train()
    total_loss = lf_rec = bp_rec = hf_rec = 0.0
    n_batches = len(loader)

    for x_lf, x_bp, x_hf in loader:
        x_lf = x_lf.to(device)
        x_bp = x_bp.to(device)
        x_hf = x_hf.to(device)

        optimizer.zero_grad()
        recons, mus, logvars, x_hat_fused = model(x_lf, x_bp, x_hf)
        targets = {'lf': x_lf, 'bp': x_bp, 'hf': x_hf}
        betas = {'lf': 1.0, 'bp': 1.0, 'hf': 1.0}
        loss, loss_dict = band_split_vae_loss(
            recons, mus, logvars, targets, x_hat_fused, None, betas=betas, alpha_fusion=0.0
        )
        loss.backward()
        optimizer.step()

        total_loss += loss_dict['total']
        lf_rec += loss_dict['recon_lf']
        bp_rec += loss_dict['recon_bp']
        hf_rec += loss_dict['recon_hf']

    return {
        'total': total_loss / n_batches,
        'lf_rec': lf_rec / n_batches,
        'bp_rec': bp_rec / n_batches,
        'hf_rec': hf_rec / n_batches,
    }

In [ ]:

best_val = float('inf')
best_path = save_dir / 'stage1_pretrained.pt'

for epoch in range(1, config.epochs + 1):
    start = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, config.device, use_amp=use_amp)
    val_loss = validate(model, val_loader, config.device)
    duration = (time.time() - start) / 60

    print(
        f"Epoch {epoch}/{config.epochs} | "
        f"train {train_loss['total']:.4f} (lf {train_loss['lf_rec']:.4f}, bp {train_loss['bp_rec']:.4f}, hf {train_loss['hf_rec']:.4f}) | "
        f"val {val_loss['total']:.4f} (lf {val_loss['lf_rec']:.4f}, bp {val_loss['bp_rec']:.4f}, hf {val_loss['hf_rec']:.4f}) | "
        f"{duration:.1f} min"
    )

    if val_loss['total'] < best_val:
        best_val = val_loss['total']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val,
            'config': config,
        }, best_path)
        print(f'Saved best to {best_path}')
